# Gold — fact_monthly_performance

`silver.monthly_performance` + `gold.dim_ticker` + `gold.bridge_ticker_manager` →
**`gold.fact_monthly_performance`**.

**Grain: one row per (ticker, month, manager).** The atomic grain, and the only fact.
Expected: **36,724 rows**.

Three CTEs, each doing one thing:

| CTE | what it does |
|---|---|
| `priced` | read Silver |
| `versioned` | attach the `dim_ticker` version **live in that month**, and the group and mandate keys |
| `exploded` | one row per manager, through the bridge |

The `versioned` join is **inner**, deliberately: it is what keeps the fact and the dimension
agreeing about who is in the universe. A ticker the dimension no longer holds produces no
rows, and the MERGE's delete arm removes any it held before.

The `exploded` join is **LEFT**, just as deliberately. **544 rows have no named manager** —
the three index tickers and `BSIF`. An inner join would silently delete the index from the
fact, and the index is the thing every trust is measured against.

In [ ]:
CREATE OR REPLACE TEMP VIEW gold_stage_fact_monthly AS
WITH priced AS (
  SELECT ticker, month_key, close, dividend, price_return, total_return, return_basis
  FROM `index-vs-trust-pipeline`.silver.monthly_performance
),
versioned AS (
  -- The whole reason the surrogate key exists: each month attaches to the version of the
  -- trust that was live that month, so "returns while X managed it" is later a plain join.
  SELECT p.*,
         d.ticker_key,
         d.manager,
         MD5(d.management_group) AS management_group_key,
         MD5(d.aic_sector)       AS mandate_key
  FROM priced p
  JOIN `index-vs-trust-pipeline`.gold.dim_ticker d
    ON d.ticker = p.ticker
   AND p.month_key BETWEEN d.effective_start_month
                       AND COALESCE(d.effective_end_month, 999912)
),
exploded AS (
  -- LEFT, not inner. The bridge holds no row for the index or for a trust with no named
  -- manager, and those months still happened. The COALESCE sends them to the special
  -- member whose name dim_ticker is already carrying.
  SELECT v.ticker, v.month_key, v.close, v.dividend, v.price_return, v.total_return,
         v.return_basis, v.ticker_key, v.management_group_key, v.mandate_key,
         COALESCE(b.manager_key, MD5(v.manager)) AS manager_key,
         COALESCE(b.manager_position, 1)         AS manager_position,
         -- 1.0 for a row with no manager: it is one whole trust-month, undivided.
         COALESCE(b.allocation_factor, 1.0)      AS allocation_factor
  FROM versioned v
  LEFT JOIN `index-vs-trust-pipeline`.gold.bridge_ticker_manager b
    ON b.ticker_key = v.ticker_key
)
SELECT MD5(CONCAT_WS('|', ticker, CAST(month_key AS STRING), manager_key)) AS monthly_key,
       ticker_key, month_key, manager_key, management_group_key, mandate_key,
       ticker, close, dividend, price_return, total_return,
       allocation_factor, manager_position, return_basis
FROM exploded;

In [ ]:
MERGE INTO `index-vs-trust-pipeline`.gold.fact_monthly_performance AS t
USING gold_stage_fact_monthly AS s
   ON t.ticker = s.ticker AND t.month_key = s.month_key AND t.manager_key = s.manager_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
-- The staging view is an inner join to dim_ticker, so a ticker that left the dimension
-- produces no rows here. Without this arm its old rows would simply stay, pointing at a
-- ticker_key that no longer exists -- a broken star rather than a smaller one. It also
-- clears the rows a manager change strands when the bridge reopens.
WHEN NOT MATCHED BY SOURCE THEN DELETE;

## Verification

Every expected answer below was measured from the warehouse **before** this notebook was
written, and is stated in `specs/11_monthly_fact/monthly-fact.md`.

In [ ]:
SELECT COUNT(*)                                       AS rows_total,
       COUNT(*) - COUNT(DISTINCT monthly_key)         AS duplicate_keys,
       COUNT(DISTINCT ticker, month_key)              AS trust_months,
       ROUND(SUM(allocation_factor), 6)               AS total_weight,
       COUNT(DISTINCT ticker)                         AS tickers,
       (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.silver.monthly_performance) AS silver_rows
FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance;

Expect **36,724 / 0 / 15,333 / 15,333 / 93 / 15,333**.

**These are the two fan-out guards.** `trust_months` must equal `silver_rows`: the fact
explodes Silver, it never loses or invents a month. And `total_weight` must equal it too —
the allocation factors sum to one whole trust-month per trust-month, which is what makes any
weighted aggregate reconcile.

If `trust_months` is below 15,333 the `versioned` join has dropped rows. If `rows_total`
is 15,333 the explode has not happened at all.

In [ ]:
-- Referential integrity on all five keys. A pure star is only pure if every join lands.
SELECT (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance f
         LEFT JOIN `index-vs-trust-pipeline`.gold.dim_ticker d ON d.ticker_key = f.ticker_key
        WHERE d.ticker_key IS NULL)                        AS orphan_ticker_keys,
       (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance f
         LEFT JOIN `index-vs-trust-pipeline`.gold.dim_manager m ON m.manager_key = f.manager_key
        WHERE m.manager_key IS NULL)                       AS orphan_manager_keys,
       (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance f
         LEFT JOIN `index-vs-trust-pipeline`.gold.dim_management_group g
           ON g.management_group_key = f.management_group_key
        WHERE g.management_group_key IS NULL)              AS orphan_group_keys,
       (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance f
         LEFT JOIN `index-vs-trust-pipeline`.gold.dim_mandate n ON n.mandate_key = f.mandate_key
        WHERE n.mandate_key IS NULL)                       AS orphan_mandate_keys,
       (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance f
         LEFT JOIN `index-vs-trust-pipeline`.gold.dim_date t ON t.month_key = f.month_key
        WHERE t.month_key IS NULL)                         AS orphan_month_keys;

Expect **0 / 0 / 0 / 0 / 0**. Any non-zero means a dimension did not load before the fact,
and the workflow's dependency edges are wrong.

In [ ]:
-- The rows with no person behind them, and the fan-out they are exempt from.
SELECT m.manager_name,
       COUNT(*)                        AS rows_carried,
       COUNT(DISTINCT f.ticker)        AS tickers,
       ROUND(MIN(f.allocation_factor), 3) AS min_weight
FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance f
JOIN `index-vs-trust-pipeline`.gold.dim_manager m ON m.manager_key = f.manager_key
WHERE m.manager_name IN ('NotApplicable', 'NoInfo')
GROUP BY m.manager_name
ORDER BY rows_carried DESC;

Expect **2 rows**: `NotApplicable` carrying **543** rows over **3** tickers (SPY, IVV, VOO)
and `NoInfo` carrying **1** over **1** (`BSIF`, whose single month is far below the 36-month
floor and so never reaches a horizon). Both at weight **1.0**.

**544 rows in total.** That is the number an inner join to the bridge would have deleted.

In [ ]:
-- The fan-out itself, so its size is a measured fact rather than an assumption.
SELECT manager_position, COUNT(*) AS rows_at_this_position
FROM `index-vs-trust-pipeline`.gold.fact_monthly_performance
GROUP BY manager_position
ORDER BY manager_position;

Position 1 must hold **15,333** rows — exactly one per trust-month, which is the third way
of stating the same guard. Every position above 1 is a duplicate month, and they total
**21,391**. The longest list runs to position **12** (NB Private Equity Partners, a fund of
funds; the length is real, not a parsing failure).

`WHERE manager_position = 1` is the cheap way to read this fact as one row per trust-month,
and `SELECT DISTINCT` on the measure columns is the legible way. `semantic.v_horizon_performance`
uses the legible one.